# Pointers

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Take an address with &, follow it with *, and use pointer parameters to change the caller's variables or return several results.
- Walk an array or a string with a pointer, and explain why a[i] is *(a + i).
- Pass a function to another function through a function pointer, as qsort does.

**In this notebook**, one idea at a time (run each cell before moving on):
1. Every variable has an address
2. A pointer holds an address
3. Changing the caller's variables
4. Pointers and arrays
5. NULL: pointing nowhere
6. const and pointers
7. Common mistakes

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## 1. Every variable has an address

Memory is a long row of numbered bytes, and each variable sits at some **address**. `&x` gives the address of `x`; `printf` shows addresses with `%p`. The actual numbers change from run to run, but the idea doesn't.

In [ ]:
int x = 5;
int y = 7;
printf("x = %d lives at %p\n", x, (void *) &x);
printf("y = %d lives at %p\n", y, (void *) &y);
printf("an int takes %zu bytes\n", sizeof x);

🧪 **Try it:** run it twice: do the addresses change? How far apart are `x` and `y`?

## 2. A pointer holds an address

A **pointer** is a variable that holds an address. `int *p = &x;` makes `p` point to `x`. Then `*p` means "the variable `p` points to": reading `*p` reads `x`, and `*p = 9;` writes into `x`. The `*` has two jobs: in a declaration it says "pointer"; in an expression it **follows** the pointer. The slides show it step by step.

#### 🎞️ A pointer, step by step

![int x = 5 lives at 0x1000; p = &x stores that address; *p = 9 writes 9 into x through the pointer](pointer-steps.svg)

In [ ]:
int x = 5;
int *p = &x;
printf("p holds %p, the address of x (%p)\n", (void *) p, (void *) &x);
printf("*p = %d\n", *p);
*p = 9;
printf("after *p = 9, x = %d\n", x);
int y = 1;
p = &y;          // p now points somewhere else
*p += 10;
printf("x = %d, y = %d\n", x, y);

🧪 **Try it:** predict the last line before running. Then create `int *q = p;`: change `*q` and print `y`.

## 3. Changing the caller's variables

Arguments are copies, so a function can't change the caller's `int`. But if it gets the variable's **address**, it can follow it with `*` and change the variable itself. That's why `scanf` takes `&qty`. A function can also "return" several results this way, through pointer parameters.

In [ ]:
#include <stdio.h>

void split_time(int total, int *minutes, int *seconds) {
    *minutes = total / 60;
    *seconds = total % 60;
}

int main(void) {
    int m;
    int s;
    split_time(135, &m, &s);
    printf("135 s = %d min %d s\n", m, s);
    return 0;
}

🧪 **Try it:** call `split_time(135, m, s)` without the `&` and read the warning: an int isn't an address.

### ✏️ Exercise: Swap two ints

Swap the values of two variables. A C function only gets copies, so it receives their addresses instead.

In [ ]:
void swap(int *a, int *b) {
    // TODO: use a temporary and *a, *b
}

int main(void) {
    int x = 1;
    int y = 2;
    swap(&x, &y);
    check_int("x is now 2", x, 2);
    check_int("y is now 1", y, 1);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
void swap(int *a, int *b) {
    int tmp = *a;
    *a = *b;
    *b = tmp;
}

int main(void) {
    int x = 1;
    int y = 2;
    swap(&x, &y);
    check_int("x is now 2", x, 2);
    check_int("y is now 1", y, 1);
    return 0;
}
```

</details>

## 4. Pointers and arrays

An array's name, used in an expression, is the **address of its first element**: `a` is `&a[0]`. Adding to a pointer moves it by whole **elements**, not bytes: `a + 1` points to `a[1]`. That's what `a[i]` really means: `*(a + i)`. A pointer can also walk an array with `p++`.

In [ ]:
int a[] = {10, 20, 30, 40};
printf("a == &a[0]? %d\n", a == &a[0]);
printf("*(a + 2) = %d, a[2] = %d\n", *(a + 2), a[2]);
int sum = 0;
for (int *p = a; p < a + 4; p++) {
    sum += *p;
}
printf("sum = %d\n", sum);

🧪 **Try it:** print `(char *) (a + 1) - (char *) a`: one step of an `int` pointer is how many bytes?

## 5. NULL: pointing nowhere

`NULL` is a pointer that points nowhere. Functions return it to say "not found" or "no memory". **Always check** for `NULL` before following a pointer with `*`: following `NULL` crashes the program.

In [ ]:
const char *text = "pointer";
char *hit = strchr(text, 't');
if (hit != NULL) {
    printf("found '%c' at index %ld\n", *hit, (long) (hit - text));
}
char *miss = strchr(text, 'z');
if (miss == NULL) {
    printf("no 'z' in \"%s\"\n", text);
}

🧪 **Try it:** print `*miss` without the check and run: the memory check reports the access to address 0.

### ✏️ Exercise: Find a value

Return a pointer to the first element equal to `target`, or NULL when it isn't there.

In [ ]:
int *find(int *a, int n, int target) {
    // TODO: return &a[i] when you find it
    return NULL;
}

int main(void) {
    int v[] = {5, 7, 9, 7};
    int *hit = find(v, 4, 7);
    check_int("finds the first 7", hit == &v[1], 1);
    check_int("missing gives NULL", find(v, 4, 4) == NULL, 1);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int *find(int *a, int n, int target) {
    for (int i = 0; i < n; i++) {
        if (a[i] == target) {
            return &a[i];
        }
    }
    return NULL;
}

int main(void) {
    int v[] = {5, 7, 9, 7};
    int *hit = find(v, 4, 7);
    check_int("finds the first 7", hit == &v[1], 1);
    check_int("missing gives NULL", find(v, 4, 4) == NULL, 1);
    return 0;
}
```

</details>

## 6. const and pointers

`const int *p` is a pointer through which you can only **read**: `*p = 3` doesn't compile. Use it in parameters to promise the function won't change the data, as `strlen(const char *s)` does. The pointer itself can still move (`p++`).

In [ ]:
#include <stdio.h>

int count_negatives(const int *values, int n) {
    int count = 0;
    for (const int *p = values; p < values + n; p++) {
        if (*p < 0) {
            count++;
        }
    }
    return count;
}

int main(void) {
    int temps[] = {3, -2, 0, -7, 5};
    printf("%d negatives\n", count_negatives(temps, 5));
    return 0;
}

🧪 **Try it:** add `*p = 0;` inside the loop and read the error.

## 7. Common mistakes

- **A pointer that was never set:** `int *p; *p = 3;` writes to a random place. Start pointers at a real address or at `NULL`.
- **Following `NULL`** without checking.
- **Returning the address of a local variable:** it disappears when the function returns, so the pointer is left pointing at nothing (the compiler warns).
- **Mixing up `p` and `*p`:** `p = 5` changes where it points; `*p = 5` changes the value there.

In [ ]:
int value = 4;
int *p = NULL;
if (p == NULL) {
    p = &value;
}
*p *= 10;
printf("value = %d\n", value);

🧪 **Try it:** change `*p *= 10;` to `p *= 10;` and read the error: you can't multiply an address.

## In short

A **pointer** holds the address of a variable. `&x` is the address of `x`; if `p` holds it, `*p` is `x` itself, to read or to change. That is how a function changes its caller's variables, and how `scanf` fills yours.

```c
#include <stdio.h>

void min_max(const int *values, int n, int *min, int *max) {
    *min = *max = values[0];
    for (const int *p = values + 1; p < values + n; p++) {
        if (*p < *min) *min = *p;
        if (*p > *max) *max = *p;
    }
}

int main(void) {
    int temps[] = {21, 17, 25, 19};
    int low, high;
    min_max(temps, 4, &low, &high);
    printf("%d to %d\n", low, high);    // 17 to 25
    return 0;
}
```

An array name is the address of its first element, so `values + 1` points at the next element and `a[i]` means `*(a + i)`.

**Watch out:** a pointer must point somewhere valid before you use `*`. Following `NULL`, or an address that was never set, crashes or corrupts memory.

## Readings

- [Pointers: Cower In Fear!](https://beej.us/guide/bgc/html/split/pointers.html) (Beej's Guide to C Programming, chapter 5, 12 min): Memory and variables, dereferencing, passing pointers as arguments (the swap you couldn't write in Functions), and the NULL pointer. Pointer arithmetic is in chapter 11 of the same guide.
- [Lecture 4: Pointers](https://cs50.harvard.edu/x/2026/notes/4/#pointers) (CS50x (Harvard), lecture notes, 15 min): Optional: from "Pointers" to "Pointer Arithmetic", then "Swapping". The pictures of addresses and arrows make the idea concrete; skip "Copying and malloc" until the next unit.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Sort Three** · Easy
- **Stats Through Pointers** · Easy
- **Reverse Words** · Medium
- **Find Every Match** · Medium
- **Multi-Key Sort** · Hard